<a href="https://colab.research.google.com/github/ismail-omar/csv/blob/main/FER2013_MobileNetV2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# 1- استيراد المكتبات
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


TensorFlow version: 2.20.0
GPU devices: []


In [ ]:

# 2- ربط Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

DATASET_ROOT = Path("/content/drive/MyDrive/Dataset FER2013")

TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"

assert TRAIN_DIR.exists(), f"لم يتم العثور على مجلد التدريب: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"لم يتم العثور على مجلد الاختبار: {TEST_DIR}"

print("Train directory:", TRAIN_DIR)
print("Test directory :", TEST_DIR)
print("Classes:", sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()]))


In [ ]:

# 3- إعدادات التدريب
IMG_SIZE = (96, 96)      # صور FER2013 أصلها صغيرة؛ سيتم تكبيرها لتناسب MobileNetV2
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.20
INITIAL_EPOCHS = 15
FINE_TUNE_EPOCHS = 8
AUTOTUNE = tf.data.AUTOTUNE


In [ ]:
# 4- تحميل البيانات وتقسيمها
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="int",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="int",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    label_mode="int",
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

assert class_names == test_ds.class_names, (
    f"ترتيب الفئات مختلف بين train وtest:\n"
    f"Train: {class_names}\nTest: {test_ds.class_names}"
)

print("Class names:", class_names)
print("Number of classes:", NUM_CLASSES)


In [ ]:

# 5- تحسين أداء خط إدخال البيانات
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


In [ ]:
# 6- عرض صور من المجموعات
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(min(16, len(images))):
        ax = plt.subplot(4, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 7- حساب عدد الصور في كل فئةوحساب أوزان الفئات
def count_images_per_class(directory, class_names):
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
    counts = {}
    labels = []

    for class_index, class_name in enumerate(class_names):
        class_dir = Path(directory) / class_name
        count = sum(
            1 for file in class_dir.rglob("*")
            if file.is_file() and file.suffix.lower() in valid_ext
        )
        counts[class_name] = count
        labels.extend([class_index] * count)

    return counts, np.asarray(labels, dtype=np.int32)

class_counts, train_labels_for_weights = count_images_per_class(TRAIN_DIR, class_names)
print("Images per class:", class_counts)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels_for_weights
)
class_weights = {i: float(w) for i, w in enumerate(weights)}

print("Class weights:", class_weights)


In [ ]:
# 8- Data Augmentation
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomTranslation(height_factor=0.08, width_factor=0.08),
        tf.keras.layers.RandomContrast(0.15),
    ],
    name="data_augmentation"
)


In [ ]:

# 9- عرض تأثير Data Augmentation
for images, _ in train_ds.take(1):
    sample = images[0:1]
    plt.figure(figsize=(12, 6))
    for i in range(8):
        augmented = data_augmentation(sample, training=True)
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(augmented[0], tf.uint8))
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# 10- بناء نموذج MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,), name="input_image")
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

model = tf.keras.Model(inputs, outputs, name="FER2013_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# 11- إعداد Callbacks وتدريب الطبقات الجديدة
OUTPUT_DIR = Path("/content/drive/MyDrive/FER2013_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_fer2013_mobilenetv2.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]


In [ ]:

history_initial = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
# 12- Fine-tuning
base_model.trainable = True

# تجميد معظم الشبكة وفتح آخر 30 طبقة فقط
fine_tune_at = max(0, len(base_model.layers) - 30)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# إبقاء BatchNormalization مجمدة يجعل Fine-tuning أكثر استقرارًا
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

total_epochs = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=len(history_initial.history["loss"]),
    epochs=total_epochs,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
# 13- دمج سجل التدريب ورسم Accuracy وLoss
def combine_histories(*histories):
    combined = {}
    for history in histories:
        for key, values in history.history.items():
            combined.setdefault(key, []).extend(values)
    return combined

history = combine_histories(history_initial, history_fine)

epochs_range = range(1, len(history["loss"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_range, history["accuracy"], label="Train Accuracy")
plt.plot(epochs_range, history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(epochs_range, history["loss"], label="Train Loss")
plt.plot(epochs_range, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 14- تحميل أفضل نسخة تم حفظها
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# 15- Confusion Matrix وClassification Report
y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

prediction_probabilities = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(prediction_probabilities, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

report_text = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)
print(report_text)


In [ ]:

# 16- حفظ التقرير بصيغة CSV وTXT
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    digits=4
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(OUTPUT_DIR / "classification_report.csv", index=True)

with open(OUTPUT_DIR / "classification_report.txt", "w", encoding="utf-8") as file:
    file.write(report_text)

np.save(OUTPUT_DIR / "confusion_matrix.npy", cm)

print("Evaluation files saved in:", OUTPUT_DIR)


In [ ]:
# 17- حفظ النموذج وتحويله إلى TFLite
FINAL_KERAS_PATH = OUTPUT_DIR / "fer2013_mobilenetv2_final.keras"
TFLITE_PATH = OUTPUT_DIR / "fer2013_mobilenetv2.tflite"

best_model.save(FINAL_KERAS_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as file:
    file.write(tflite_model)

print("Keras model saved to:", FINAL_KERAS_PATH)
print("TFLite model saved to:", TFLITE_PATH)
print("TFLite size (MB):", round(TFLITE_PATH.stat().st_size / (1024 ** 2), 2))


In [ ]:
#18- اختبار نموذج TFLite على صورة واحدة
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details)
print("Output details:", output_details)

for batch_images, batch_labels in test_ds.take(1):
    sample_image = batch_images[0:1].numpy().astype(input_details[0]["dtype"])
    true_label = int(batch_labels[0].numpy())

interpreter.set_tensor(input_details[0]["index"], sample_image)
interpreter.invoke()

tflite_output = interpreter.get_tensor(output_details[0]["index"])
predicted_label = int(np.argmax(tflite_output[0]))

plt.imshow(sample_image[0].astype("uint8"))
plt.title(
    f"True: {class_names[true_label]} | "
    f"Predicted: {class_names[predicted_label]}"
)
plt.axis("off")
plt.show()
